<a href="https://colab.research.google.com/github/leeeshart/PromptSentinel/blob/main/notebook3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PromptSentinel — Notebook 3
## Long Prompt & Human-Written Jailbreak Analysis

**Author:** Leesha Mogha

**Institution:** IMS Ghaziabad (University Course Campus)

**Project:** PromptSentinel (v4)

**Builds on:** [Prompt-Safety-Classifier](https://github.com/leeeshart/Prompt-Safety-Classifier)

---

### What this notebook does
Tests whether the v4 model maintains its performance on longer,
human-written jailbreaks where harmful content is buried in
creative text — the failure mode Notebook 2 could not test
because WildJailbreak prompts are synthetically generated.

### Research question answered here
**RQ2: Does performance drop on longer, human-written jailbreaks?**

In [1]:
!pip install datasets pandas numpy scikit-learn matplotlib seaborn -q

from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score
)
import warnings
warnings.filterwarnings('ignore')

login()

print("Libraries loaded!")

Libraries loaded!


---
## Section 2: Load Data

Two data sources needed for this notebook:

1. **Combined dataset** (from Notebook 1) — used to train the v4 model
2. **TrustAIRLab** — reloaded separately as the human-written jailbreak test set

We keep TrustAIRLab completely out of training so it serves
as a clean out-of-distribution test. This is the

In [2]:
# Upload combined_dataset_final.csv from Notebook 1
from google.colab import files
import pandas as pd

print("Please upload compressed_data.csv.gz")
uploaded = files.upload()

df_train = pd.read_csv('compressed_data.csv.gz')

print(f"Combined dataset loaded: {len(df_train):,} prompts")
print(df_train['label'].value_counts())
print()
print("Sources:")
print(df_train['source'].value_counts())

Please upload compressed_data.csv.gz


Saving compressed_data.csv.gz to compressed_data.csv (1).gz
Combined dataset loaded: 116,202 prompts
label
safe      63142
unsafe    53060
Name: count, dtype: int64

Sources:
source
wildjailbreak    100099
trustairlab        6142
toxicchat          4981
qualifire          4980
Name: count, dtype: int64


### Removing TrustAIRLab from training data

TrustAIRLab will be used as the held-out human-written test set.
It must be completely excluded from training to avoid data leakage.

In [5]:
# Remove TrustAIRLab from training data
df_train = df_train[df_train['source'] != 'trustairlab'].reset_index(drop=True)

print(f"Training set after removing TrustAIRLab: {len(df_train):,} prompts")
print(df_train['label'].value_counts())
print()
print("Sources remaining in training:")
print(df_train['source'].value_counts())

Training set after removing TrustAIRLab: 110,060 prompts
label
safe      57653
unsafe    52407
Name: count, dtype: int64

Sources remaining in training:
source
wildjailbreak    100099
toxicchat          4981
qualifire          4980
Name: count, dtype: int64


### TrustAIRLab — Human-written jailbreak test set

Reloading TrustAIRLab separately and keeping it completely
outside of training. These are real human-written jailbreaks
collected from Reddit and Discord — not synthetic wraps.

In [6]:
from datasets import load_dataset
# Load TrustAIRLab as the held-out human-written test set
jailbreak = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'jailbreak_2023_05_07',
    split='train'
)
regular = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'regular_2023_05_07',
    split='train'
)

df_unsafe = jailbreak.to_pandas()[['prompt']]
df_safe   = regular.to_pandas()[['prompt']]

df_unsafe['label']  = 'unsafe'
df_safe['label']    = 'safe'

df_test_human = pd.concat([df_unsafe, df_safe], ignore_index=True)
df_test_human = df_test_human.dropna()
df_test_human = df_test_human[df_test_human['prompt'].str.strip() != '']

print(f"TrustAIRLab test set loaded: {len(df_test_human):,} prompts")
print(df_test_human['label'].value_counts())

TrustAIRLab test set loaded: 6,387 prompts
label
safe      5721
unsafe     666
Name: count, dtype: int64


In [7]:
# Section 3: Train v4 Model

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# Shuffle training data
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

vectorizer_v4 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train = vectorizer_v4.fit_transform(df_train['prompt'])
y_train = df_train['label']

model_v4 = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
model_v4.fit(X_train, y_train)

print("v4 model trained.")
print(f"  Training samples : {len(df_train):,}")
print(f"  Vocabulary size  : {len(vectorizer_v4.vocabulary_):,}")

v4 model trained.
  Training samples : 110,060
  Vocabulary size  : 10,000


In [8]:
# Section 4: Test on Human-Written Jailbreaks (TrustAIRLab)

X_test_human = vectorizer_v4.transform(df_test_human['prompt'])
y_test_human  = df_test_human['label']

y_pred_human = model_v4.predict(X_test_human)

print("=== v4 on TrustAIRLab (human-written jailbreaks) ===")
print()
print(classification_report(y_test_human, y_pred_human, zero_division=0))

from sklearn.metrics import recall_score, precision_score, f1_score
recall_v4_human    = recall_score(y_test_human, y_pred_human, pos_label='unsafe', zero_division=0)
precision_v4_human = precision_score(y_test_human, y_pred_human, pos_label='unsafe', zero_division=0)
f1_v4_human        = f1_score(y_test_human, y_pred_human, pos_label='unsafe', zero_division=0)

print(f"Unsafe recall    : {recall_v4_human:.3f}")
print(f"Unsafe precision : {precision_v4_human:.3f}")
print(f"Unsafe F1        : {f1_v4_human:.3f}")

=== v4 on TrustAIRLab (human-written jailbreaks) ===

              precision    recall  f1-score   support

        safe       0.94      0.64      0.76      5721
      unsafe       0.17      0.65      0.28       666

    accuracy                           0.64      6387
   macro avg       0.56      0.65      0.52      6387
weighted avg       0.86      0.64      0.71      6387

Unsafe recall    : 0.653
Unsafe precision : 0.174
Unsafe F1        : 0.275


In [9]:
# Section 5: Prompt Length vs Recall Analysis

import numpy as np
import matplotlib.pyplot as plt

df_test_human = df_test_human.copy()
df_test_human['pred']   = y_pred_human
df_test_human['length'] = df_test_human['prompt'].str.split().str.len()

# Focus on unsafe prompts — where misses matter most
df_unsafe_test = df_test_human[df_test_human['label'] == 'unsafe'].copy()

# Bin by length
bins   = [0, 50, 100, 200, 400, 800, 9999]
labels = ['0–50', '51–100', '101–200', '201–400', '401–800', '800+']
df_unsafe_test['length_bin'] = pd.cut(df_unsafe_test['length'], bins=bins, labels=labels)

recall_by_length = (
    df_unsafe_test
    .groupby('length_bin', observed=True)
    .apply(lambda g: (g['pred'] == 'unsafe').sum() / len(g))
    .reset_index()
)
recall_by_length.columns = ['length_bin', 'recall']

count_by_length = df_unsafe_test['length_bin'].value_counts().sort_index()

print("Unsafe recall by prompt length:")
print()
for _, row in recall_by_length.iterrows():
    n = count_by_length.get(row['length_bin'], 0)
    bar = '█' * int(row['recall'] * 30)
    print(f"  {str(row['length_bin']):>10} words  |{bar:<30}| {row['recall']:.2f}  (n={n})")

Unsafe recall by prompt length:

        0–50 words  |█████████████████             | 0.57  (n=44)
      51–100 words  |██████████████                | 0.49  (n=76)
     101–200 words  |█████████████████             | 0.58  (n=150)
     201–400 words  |█████████████████████         | 0.72  (n=187)
     401–800 words  |█████████████████████         | 0.72  (n=163)
        800+ words  |█████████████████████         | 0.72  (n=46)


In [10]:
# Section 6: v2 vs v4 Head-to-Head Comparison
# v2 used TrustAIRLab only — recreate that here for a fair comparison

jailbreak_v2 = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'jailbreak_2023_05_07', split='train'
)
regular_v2 = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'regular_2023_05_07', split='train'
)
df_v2_unsafe = jailbreak_v2.to_pandas()[['prompt']]
df_v2_safe   = regular_v2.to_pandas()[['prompt']]
df_v2_unsafe['label'] = 'unsafe'
df_v2_safe['label']   = 'safe'
df_v2 = pd.concat([df_v2_unsafe, df_v2_safe], ignore_index=True).sample(frac=1, random_state=0)

# Simple 80/20 split as v2 used
train_v2, test_v2 = train_test_split(df_v2, test_size=0.2, random_state=42, stratify=df_v2['label'])

vectorizer_v2 = TfidfVectorizer(max_features=5000)
model_v2 = LogisticRegression(max_iter=1000, class_weight='balanced')
model_v2.fit(vectorizer_v2.fit_transform(train_v2['prompt']), train_v2['label'])

# Both models tested on the SAME TrustAIRLab test prompts
y_pred_v2 = model_v2.predict(vectorizer_v2.transform(df_test_human['prompt']))

recall_v2    = recall_score(y_test_human, y_pred_v2, pos_label='unsafe', zero_division=0)
precision_v2 = precision_score(y_test_human, y_pred_v2, pos_label='unsafe', zero_division=0)
f1_v2        = f1_score(y_test_human, y_pred_v2, pos_label='unsafe', zero_division=0)

comparison = pd.DataFrame([
    {
        'Model'    : 'v2 (TrustAIRLab only, 6k prompts)',
        'Recall'   : round(recall_v2, 3),
        'Precision': round(precision_v2, 3),
        'F1'       : round(f1_v2, 3),
    },
    {
        'Model'    : 'v4 (4 sources, 110k prompts)',
        'Recall'   : round(recall_v4_human, 3),
        'Precision': round(precision_v4_human, 3),
        'F1'       : round(f1_v4_human, 3),
    },
])

print("=== v2 vs v4 on human-written jailbreaks ===")
print()
print(comparison.to_string(index=False))
print()
delta_recall = round(recall_v4_human - recall_v2, 3)
print(f"Recall change v2 → v4: {delta_recall:+.3f}")

=== v2 vs v4 on human-written jailbreaks ===

                            Model  Recall  Precision    F1
v2 (TrustAIRLab only, 6k prompts)   0.962      0.680 0.797
     v4 (4 sources, 110k prompts)   0.653      0.174 0.275

Recall change v2 → v4: -0.309


In [12]:
# Section 7: Results Summary — Answer RQ2

print("=" * 50)
print("NOTEBOOK 3 SUMMARY — RQ2")
print("=" * 50)
print()
print("RQ2: Does performance drop on longer, human-written jailbreaks?")
print()

shortest_recall = recall_by_length[recall_by_length['length_bin'] == '0–50']['recall'].values
longest_recall  = recall_by_length[recall_by_length['length_bin'].isin(['401–800', '800+'])]['recall'].values

if len(shortest_recall) and len(longest_recall):
    short_val = float(shortest_recall[0])
    long_val  = float(longest_recall.mean())
    diff      = round(long_val - short_val, 3)

    print(f"  Recall on short prompts (0–50 words) : {short_val:.3f}")
    print(f"  Recall on long prompts  (400+ words) : {long_val:.3f}")
    print(f"  Difference (long − short)            : {diff:+.3f}")
    print()
    print("  Finding: Recall is HIGHER on longer prompts, not lower.")
    print("  Interpretation: Long jailbreaks contain more harmful vocabulary,")
    print("  giving TF-IDF more signal. Short jailbreaks are harder because")
    print("  harmful intent is compressed into fewer, more creative words.")
    print()
    print("  RQ2 answer: Length is not the bottleneck. Vocabulary mismatch is.")
    print("  The hard cases are short, creative, human-written jailbreaks")
    print("  that use indirect language not present in synthetic training data.")

print()
print("-" * 50)
print("v2 vs v4 comparison — important caveat")
print("-" * 50)
print()
print(f"  v2 recall on TrustAIRLab: {recall_v2:.3f}  (trained on same distribution)")
print(f"  v4 recall on TrustAIRLab: {recall_v4_human:.3f}  (true out-of-distribution test)")
print()
print("  v2's higher recall is an in-distribution artifact, not a genuine")
print("  advantage. v4 was never trained on TrustAIRLab-style prompts.")
print("  The v4 number is the honest out-of-distribution baseline.")
print()
print("  What this exposes: even 110k training prompts dominated by synthetic")
print("  data (WildJailbreak) do not generalise well to human-written jailbreaks.")
print("  This is the threat model mismatch the paper discusses.")
print()
print("Next: Notebook 4 — Chunked Embedding Experiment")
print("  The short-prompt recall problem may be addressable with")
print("  meaning-based embeddings rather than vocabulary-based TF-IDF.")

NOTEBOOK 3 SUMMARY — RQ2

RQ2: Does performance drop on longer, human-written jailbreaks?

  Recall on short prompts (0–50 words) : 0.568
  Recall on long prompts  (400+ words) : 0.721
  Difference (long − short)            : +0.152

  Finding: Recall is HIGHER on longer prompts, not lower.
  Interpretation: Long jailbreaks contain more harmful vocabulary,
  giving TF-IDF more signal. Short jailbreaks are harder because
  harmful intent is compressed into fewer, more creative words.

  RQ2 answer: Length is not the bottleneck. Vocabulary mismatch is.
  The hard cases are short, creative, human-written jailbreaks
  that use indirect language not present in synthetic training data.

--------------------------------------------------
v2 vs v4 comparison — important caveat
--------------------------------------------------

  v2 recall on TrustAIRLab: 0.962  (trained on same distribution)
  v4 recall on TrustAIRLab: 0.653  (true out-of-distribution test)

  v2's higher recall is an in-dist